In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
from datasets import load_dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
from trl import setup_chat_format

model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

In [ ]:
dataset = load_dataset('Amod/mental_health_counseling_conversations')

README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Context', 'Response'],
        num_rows: 3512
    })
})

In [ ]:
def tokenize_text(ds):
    ds['text'] = tokenizer.apply_chat_template([{'role': 'user', 'content':ds['Context'].strip()}, {'role': 'assistant', 'content':ds['Response'].strip()}], tokenize=False)
    return ds

In [ ]:
print(tokenize_text(dataset['train'][0]))

{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?", 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is someho

In [ ]:
# dataset = dataset.train_test_split(test_size=0.2, shuffle=True)

In [ ]:
tokenized_dataset = dataset.map(tokenize_text)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['Context', 'Response', 'text'],
        num_rows: 3512
    })
})

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(['Context', 'Response'])

In [ ]:
# from datasets import
tokenized_dataset = tokenized_dataset['train'].train_test_split(test_size=0.2, shuffle=True)

In [ ]:
tokenized_dataset['train']

Dataset({
    features: ['text'],
    num_rows: 2809
})

In [ ]:
!pip install trl

from trl import SFTConfig, SFTTrainer

In [ ]:
# Configure trainer
training_args = SFTConfig(
    output_dir="./sft_output_new",
    max_steps=2000,
    per_device_train_batch_size=4,
    # gradient_accumulation_steps=8,
    learning_rate=1e-5,
    logging_steps=50,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=100,
)

In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
)

In [ ]:
torch.cuda.empty_cache()

trainer.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.721900,2.696771,2.697499,117444.000000,0.426366
200,2.629400,2.650134,2.661725,237463.000000,0.431064
300,2.648700,2.620462,2.630585,354497.000000,0.433910
400,2.604800,2.598260,2.596305,475946.000000,0.436488
500,2.585300,2.580321,2.590554,591309.000000,0.439518
600,2.543200,2.565869,2.578247,711253.000000,0.444212
700,2.508400,2.552179,2.552301,827559.000000,0.446101
800,2.486700,2.540689,2.523348,943293.000000,0.448503
900,2.465700,2.530434,2.517346,1060769.000000,0.450039
1000,2.440300,2.521831,2.493137,1175961.000000,0.451564


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.721900,2.696771,2.697499,117444.000000,0.426366
200,2.629400,2.650134,2.661725,237463.000000,0.431064
300,2.648700,2.620462,2.630585,354497.000000,0.433910
400,2.604800,2.598260,2.596305,475946.000000,0.436488
500,2.585300,2.580321,2.590554,591309.000000,0.439518
600,2.543200,2.565869,2.578247,711253.000000,0.444212
700,2.508400,2.552179,2.552301,827559.000000,0.446101
800,2.486700,2.540689,2.523348,943293.000000,0.448503
900,2.465700,2.530434,2.517346,1060769.000000,0.450039
1000,2.440300,2.521831,2.493137,1175961.000000,0.451564


TrainOutput(global_step=2000, training_loss=2.501108558654785, metrics={'train_runtime': 4148.0467, 'train_samples_per_second': 1.929, 'train_steps_per_second': 0.482, 'total_flos': 2449403228799360.0, 'train_loss': 2.501108558654785, 'epoch': 2.844950213371266})

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Use with caution: this command will permanently delete the specified directory and its contents.
!rm -r /content/sft_output/checkpoint-900

In [ ]:
prompt = "What is the primary function of mitochondria within a cell?"

In [ ]:
pipe_base = pipeline('text-generation', model=model, tokenizer=tokenizer)
pipe_base(prompt, max_new_tokens=100)

Device set to use cuda:0


[{'generated_text': 'What is the primary function of mitochondria within a cell?\nThe primary function of mitochondria within a cell is to provide energy to the cell.\xa0 Mitochondria are the sites of oxidative phosphorylation, the process by which cellular energy is generated.\xa0 The main products of oxidative phosphorylation are ATP, NADH, and FADH, which are important metabolites of cellular metabolism.\xa0 The energy-rich nature of mitochondria make them the primary site for energy production.\xa0 Mitochondria are the site of energy production and are the source of the most energy'}]

In [ ]:
pipe = pipeline('text-generation', model='/content/sft_output_new/checkpoint-2000', tokenizer=tokenizer)
pipe(prompt, max_new_tokens=100)

Device set to use cuda:0


[{'generated_text': 'What is the primary function of mitochondria within a cell?\n\nMolecules that are either broken down by enzymes that are released from the mitochondria during cell respiration or that are used by the cell as a source of energy.\n\nWhat do mitochondria do in aerobic respiration?\n\nOxidative phosphorylation in the mitochondria uses oxygen to break up ATP molecules and release electrons from water molecules.\n\nWhat does the role of the mitochondria in oxidative phosphorylation in aerobic respiration?\n\nThe mitochondria’s main role in the catabolic process of oxidative phosphory'}]